# RAG Self-Consistency
LLM의 확률적 특성을 이용해서, 여러번 답변을 생성하고, 그중에 가장 일관된 답변(다수결)을 채택해서 최종응답으로 사용하는 기법이다.

In [2]:
%pip install sentence_transformers scikit-learn -Uqqq

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

In [4]:
# 가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query=None):
    return [
        Document(page_content="파리의 상징은 에펠탑이며, 1889년에 세워졌습니다."),  # 에펠탑 기본 정보
        Document(page_content="파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다."),  # 도시 구조 및 대표 박물관
        Document(page_content="파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.")  # 관광 규모 및 랜드마크
    ]

retrieve_vectordb('파리는 안돼')

[Document(metadata={}, page_content='파리의 상징은 에펠탑이며, 1889년에 세워졌습니다.'),
 Document(metadata={}, page_content='파리는 세느강을 따라 발달한 도시로, 루브르 박물관은 파리의 상징입니다.'),
 Document(metadata={}, page_content='파리는 연간 약 2천만 명의 관광객이 방문하는 세계적 관광 도시입니다. 많은 관광객이 파리의 상징인 개선문을 방문하고 있습니다.')]

In [7]:
# 채팅 프롬프트 / 휴먼 메시지 템플릿
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser # 출력 -> 문자열 파싱

# 함수를 Runnable로 감싸 chain에서 실행
from langchain_core.runnables import RunnableLambda

# 한 번 요청으로 5개의 응답 생성
llm = init_chat_model('gpt-5.6-luna', n=5) 

prompt = PromptTemplate.from_template(''' 
아래 주어진 문서를 참고해서 사용자의 [질문]에 대한 여행일정을 작성해주세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
- 답변은 **최종추천일정:**으로 시작하세요.
- 일자별 일정은 한문장으로 요약하세요.
- 불필요한 서술은 생략하고, 핵심일정만 나열하세요.
''')

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

question = '파리의 역사, 관광지, 방문 시기를 종합하여 3일 여행 일정을 추천해주세요.'
retrieved_docs = retrieve_vectordb(question)
# 문서 본문만 뽑아서 하나의 문자열 context로 생성
context = '\n\n'.join( [doc.page_content for doc in retrieved_docs])

messages = prompt.format_prompt(context=context, question=question).to_messages()
response = llm.generate([messages])
print(response)

generations=[[ChatGeneration(text='최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  \n- **1일차:** 루브르 박물관에서 파리의 역사와 예술을 감상한 뒤 세느강을 따라 산책합니다.  \n- **2일차:** 에펠탑을 관람하고 주변에서 파리의 도시 경관과 세느강 풍경을 즐깁니다.  \n- **3일차:** 개선문을 방문해 파리의 상징적 기념물을 둘러보고 샹젤리제 거리를 관광합니다.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  \n- **1일차:** 루브르 박물관에서 파리의 역사와 예술을 감상한 뒤 세느강을 따라 산책합니다.  \n- **2일차:** 에펠탑을 관람하고 주변에서 파리의 도시 경관과 세느강 풍경을 즐깁니다.  \n- **3일차:** 개선문을 방문해 파리의 상징적 기념물을 둘러보고 샹젤리제 거리를 관광합니다.', additional_kwargs={'refusal': None}, response_metadata={'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0602c-d48f-7473-bf25-cbe3bc928cc0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 1123, 'total_tokens': 1328, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 424}})), ChatGeneration(text='최종추천일정:  \n

In [9]:
from pprint import pprint
pprint(response.generations[0])

[ChatGeneration(text='최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  \n- **1일차:** 루브르 박물관에서 파리의 역사와 예술을 감상한 뒤 세느강을 따라 산책합니다.  \n- **2일차:** 에펠탑을 관람하고 주변에서 파리의 도시 경관과 세느강 풍경을 즐깁니다.  \n- **3일차:** 개선문을 방문해 파리의 상징적 기념물을 둘러보고 샹젤리제 거리를 관광합니다.', generation_info={'finish_reason': 'stop', 'logprobs': None}, message=AIMessage(content='최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  \n- **1일차:** 루브르 박물관에서 파리의 역사와 예술을 감상한 뒤 세느강을 따라 산책합니다.  \n- **2일차:** 에펠탑을 관람하고 주변에서 파리의 도시 경관과 세느강 풍경을 즐깁니다.  \n- **3일차:** 개선문을 방문해 파리의 상징적 기념물을 둘러보고 샹젤리제 거리를 관광합니다.', additional_kwargs={'refusal': None}, response_metadata={'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0602c-d48f-7473-bf25-cbe3bc928cc0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 205, 'output_tokens': 1123, 'total_tokens': 1328, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 424}})),
 ChatGeneration(text='최종추천일정:  \n1일차: 봄·가을의 쾌

In [11]:
candidates = [gen.text for gen in response.generations[0]]
for i, cand in enumerate(candidates):
    print(f"{i+1} : {cand}")
    print()


1 : 최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  
- **1일차:** 루브르 박물관에서 파리의 역사와 예술을 감상한 뒤 세느강을 따라 산책합니다.  
- **2일차:** 에펠탑을 관람하고 주변에서 파리의 도시 경관과 세느강 풍경을 즐깁니다.  
- **3일차:** 개선문을 방문해 파리의 상징적 기념물을 둘러보고 샹젤리제 거리를 관광합니다.

2 : 최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 파리를 방문해 세느강 산책과 루브르 박물관 관람으로 파리의 역사와 문화를 체험하세요.  
2일차: 에펠탑을 중심으로 샹드마르스와 세느강 주변을 둘러보며 1889년 건립된 파리의 상징을 감상하세요.  
3일차: 개선문과 주변 명소를 방문하며 파리의 도시 발전과 역사적 의미를 살펴보고 시내 관광으로 일정을 마무리하세요.

3 : 최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 세느강을 따라 산책하며 루브르 박물관에서 파리의 역사와 예술을 감상합니다.  
2일차: 1889년에 세워진 에펠탑을 방문해 파리의 상징을 둘러보고 주변 세느강 풍경을 즐깁니다.  
3일차: 개선문을 방문해 파리의 대표 관광지를 둘러본 뒤 도심을 산책하며 여행을 마무리합니다.

4 : 최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 세느강을 따라 산책하며 루브르 박물관과 파리의 역사적 중심지를 둘러보세요.  
2일차: 1889년에 세워진 에펠탑을 방문하고 주변에서 파리의 대표적인 도시 경관을 감상하세요.  
3일차: 개선문을 방문해 파리의 상징적인 관광지를 둘러본 뒤 세느강변에서 여행을 마무리하세요.

5 : 최종추천일정:  
- 방문 시기: 봄 또는 가을의 비교적 쾌적한 계절을 추천합니다.  
- 1일차: 세느강을 따라 파리의 역사적 분위기를 둘러보고 루브르 박물관을 관람합니다.  
- 2일차: 1889년에 세워진 에펠탑을 방문해 파리의 상징적인 경관을 감상합니다.  
- 3일차: 파리의 대표 명소인 개선문을 둘러보며 세계적인 관광 도시의 분

## n개의 응답을 하나로 추출하기

In [17]:
# 출력 결과를 원하는 형식으로 변환용 기본 Parser
from langchain_core.output_parsers import BaseOutputParser  
from sentence_transformers import SentenceTransformer # 임베딩 모델
from pydantic import Field # 클래스 속성 정의용/검증용
from sklearn.cluster import KMeans # 클러스터링 모델
from collections import Counter
import numpy as np

class RobustSelfConsistencyParser(BaseOutputParser):
    n_clusters: int = Field(default=2) # 후보 그룹 갯수
    # 임베딩 모델
    encoder: object = Field(default=SentenceTransformer('all-MiniLM-L6-v2')) 

    def parse(self, generations: list[str]) -> str:
        # 1. 임베딩
        embeddings = self.encoder.encode(generations) # 벡터로 변환
        #print(embeddings.shape) # (후보 답변 갯수, 임베딩 벡터 차원)

        # 2. 클러스터링(KMeans)
        kmeans = KMeans(n_clusters=self.n_clusters,random_state=42)
        kmeans.fit(embeddings) # 클러스터링 실행
        #print(kmeans.labels_)     # 각 답변이 어느 라벨에 속해있는지 확인

        # 3. 다수결 투표
        counts = Counter(kmeans.labels_)
        # 가장 많은 후보 답변이 속한 클러스터
        target_label = max(counts, key=counts.get) 
        # 선택된 클러스터에 속한 후보 답변 인덱스만 추출
        target_indices = np.where(kmeans.labels_ == target_label)[0]
        #print(target_label)
        #print(target_indices)

        # 4. 대표 답변 선택 ( 중심점에 가장 가까운 후보 )
        target_centroid = kmeans.cluster_centers_[target_label] # 선택된 클러스터 중심점 벡터
        # 후보 답변들과 중심점 사이의 거리를 계산
        distances = np.linalg.norm(embeddings[target_indices] - target_centroid, axis = 1)
        representive_idx = np.argmin(distances) # 중심점과 가장 가까운 답변 인덱스
        # 클러스터 중 가장 대표인 답변 반환
        return generations[target_indices[representive_idx]] 

parser = RobustSelfConsistencyParser()
final_answer = parser.parse(candidates) # 후보 중 대표 답변 선택

print(f"최종 답변: {final_answer}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

최종 답변: 최종추천일정:  
1일차: 봄·가을의 쾌적한 시기에 세느강을 따라 산책하며 루브르 박물관에서 파리의 역사와 예술을 감상합니다.  
2일차: 1889년에 세워진 에펠탑을 방문해 파리의 상징을 둘러보고 주변 세느강 풍경을 즐깁니다.  
3일차: 개선문을 방문해 파리의 대표 관광지를 둘러본 뒤 도심을 산책하며 여행을 마무리합니다.


In [ ]:
# 여행 후보 일정을 여러개 생성 후, 클러스터링 기반 Self-Consistency로 최종 답변 생성하는 함수
def travel_planner(question, vervose=False):

    retrieved_docs = retrieve_vectordb(question)
    context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

    # 프롬프트 템플릿을 메시지 형태로 변환
    messages = prompt.format_prompt(context = context, question=question).to_messages()
    response = llm.generate([messages]) # n=5 로 5개 답변 생성
    candidates=[gen.text for gen in response.generations[0]] # 후보 텍스트들만 추출
    # 눈으로 후보 확인하고 싶으면 verbose = True
    if vervose:
        for i, cand in enumerate(candidates):
            print(f"{i+1}: {cand}")
            print()

    parser = RobustSelfConsistencyParser()        # 대표 답변 고르는 파서
    return parser.parse(candidates)  # 최종 대표 답변 반환

question = '파리의 역사, 관광지, 방문시기를 종합하여 3일 여행 일정을 추천해주세요.' 
travel_planner(question, vervose=True)
print(f"최종 답변 : {response}")


1: 최종추천일정:  
1일차: 에펠탑과 세느강을 둘러보며 1889년 건립된 파리의 상징과 도시의 역사적 풍경을 감상합니다.  
2일차: 루브르 박물관을 관람한 뒤 주변 역사 지구와 개선문을 방문합니다.  
3일차: 봄·가을의 쾌적한 시기에 파리 시내를 여유롭게 산책하며 주요 명소를 재방문하고 세느강 야경을 즐깁니다.

2: 최종추천일정: 방문시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  
1일차: 세느강변을 산책하며 루브르 박물관을 관람하고 파리의 역사와 문화를 체험합니다.  
2일차: 에펠탑을 방문해 1889년부터 이어진 파리의 상징을 감상한 뒤 주변을 둘러봅니다.  
3일차: 개선문과 샹젤리제 거리를 방문하며 파리의 역사적 명소와 도시 경관을 즐깁니다.

3: 최종추천일정: 방문 시기는 날씨가 쾌적하고 관광객이 비교적 적은 봄·가을을 추천합니다.  
- 1일차: 루브르 박물관에서 파리의 역사와 예술을 감상한 뒤 세느강 주변을 산책하고 노트르담 대성당을 둘러봅니다.  
- 2일차: 에펠탑을 관람하고 샹드마르스 공원과 개선문·샹젤리제 거리를 방문합니다.  
- 3일차: 베르사유 궁전 또는 몽마르트르를 둘러본 뒤 세느강 유람선에서 파리의 주요 명소를 감상합니다.

4: 최종추천일정:

- **1일차:** 봄·가을 오전에 에펠탑을 방문한 뒤 세느강을 따라 산책하며 파리의 도시 형성과 1889년 에펠탑 건립 역사를 둘러봅니다.
- **2일차:** 오전부터 루브르 박물관을 관람하고 인근 역사 지구와 세느강변을 탐방합니다.
- **3일차:** 비교적 한산한 이른 아침에 개선문과 샹젤리제 거리를 방문하며 세계적 관광 도시 파리의 대표 명소를 감상합니다.

5: 최종추천일정:

- **1일차:** 봄·가을 오전에 에펠탑을 방문한 뒤 세느강변을 산책하며 파리의 도시 발전과 1889년 건립 역사를 둘러보세요.
- **2일차:** 루브르 박물관에서 파리의 역사와 예술을 감상한 후 주변 고 historic streets? Need Korean. 개선문. 